In [ ]:
from libs import circuits, dataloading, qnn_models
import matplotlib.pyplot as plt
import numpy as np
import json
import csv
import json
import os
import pandas as pd
import keras
from keras import Sequential
from keras.layers import Dense,BatchNormalization, Dropout
import tensorflow as tf
import random
from keras.activations import relu, tanh
from keras.layers import Input, Dense
import seaborn as sns
import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score, r2_score, mean_squared_error
from sklearn.datasets import make_regression
from sklearn.utils import shuffle

plt.style.use('./libs/notebookstyle_aps.mplstyle')

%load_ext autoreload
%autoreload 2

In [ ]:
jnp_train_inputs, jnp_val_inputs, jnp_train_outputs, jnp_val_outputs = dataloading.load_data(0, n_samples = 'all')

In [ ]:
X_train_tensor = tf.convert_to_tensor(jnp_train_inputs, dtype=tf.float32)
X_test_tensor = tf.convert_to_tensor(jnp_val_inputs, dtype=tf.float32)
y_train_tensor = tf.convert_to_tensor(jnp_train_outputs, dtype=tf.float32)
y_test_tensor = tf.convert_to_tensor(jnp_val_outputs , dtype=tf.float32)

### generate classical models with random layer sizes
#### run once and select models with reasonable sizes i.e. not to many very large ones

### load and train the models

In [ ]:
save_dir_cl = './cl_results/'
save_dir_cl_models='./cl_models/'

In [ ]:
model_list_all = !ls './cl_models/'

In [ ]:
# select one instance of each model architecture out of the trained models
model_list = [s for s in model_list_all if (s[-2:] == '_1')]
models = []
n_params = []
for model1 in model_list:
    model = tf.keras.models.load_model(save_dir_cl_models + model1)
    models += [model]
    n_params += [[model.name, model.count_params()]]
    #print(model.summary())

In [ ]:
# train 10 instances of each model, letting keras randomly select different initial parameters
# save the performance metrics and history
for j in range(1,11):
    for i,my_model in enumerate(models):
        print(j, my_model.name)
        
        if (not my_model.name + '_' + str(j) in model_list_all):
            model = tf.keras.models.clone_model(my_model)
            optimizer = keras.optimizers.Adam(learning_rate=0.002)
            model.compile(optimizer=optimizer, loss='mse')
            
            starttime = time.time()
            print(' ******** Starting training ')
            callback = keras.callbacks.EarlyStopping(monitor='loss',patience=3)
            hist = model.fit(X_train_tensor, y_train_tensor, verbose=0, epochs=300, batch_size = 50, 
                             validation_data = (X_test_tensor, y_test_tensor), callbacks = [callback])
            endtime = time.time()
            print(' ')
            print(' ******** Training took: ', endtime - starttime)
        
            model.save("./cl_models/%s_%i"%(model.name,j))
                
            plt.figure()
            plt.plot(hist.history['loss'])
            plt.plot(hist.history['val_loss'])
            plt.show()
            
            ### predict
            y_train_hat = model.predict(X_train_tensor)
            y_test_hat = model.predict(X_test_tensor)
            
            y_train_hat_uz = dataloading.unpreprocess_target(y_train_hat)
            y_test_hat_uz = dataloading.unpreprocess_target(y_test_hat)
            
            y_train_uz = dataloading.unpreprocess_target(y_train_tensor)
            y_test_uz = dataloading.unpreprocess_target(y_test_tensor)
                    
            metrics ={
                'r2_train_scaled': r2_score(y_train_tensor.numpy(), np.array(y_train_hat)),
                'r2_val_scaled':  r2_score(y_test_tensor.numpy(), np.array(y_test_hat)),
                'mse_train':  mean_squared_error(y_train_tensor.numpy(), np.array(y_train_hat)),
                'mse_val':  mean_squared_error(y_test_tensor.numpy(), np.array(y_test_hat)),
                'r2_train': r2_score(np.array(y_train_uz), np.array(y_train_hat_uz)),
                'r2_val': r2_score(np.array(y_test_uz), np.array(y_test_hat_uz)) 
            }
        
            os.makedirs(save_dir_cl, exist_ok=True)
            np.savez_compressed(f"{save_dir_cl}/{model.name}_history_{j}.npz", history=hist)
            with open(f"{save_dir_cl}/clNN_r2.txt", mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([j, model.name, metrics['r2_train'], metrics['r2_val'], metrics['r2_train_scaled'], metrics['r2_val_scaled'], metrics['mse_train'], metrics['mse_val']])
        else:
            print('exists')
    